# Grammar Scorer

Zero-shot pipeline: T5 grammar correction + LanguageTool spelling detection.

**Inputs:**
- `accepted_rejected.csv` — labeled caption dataset

**Outputs saved to `/kaggle/working`:**
- `grammar_scores.csv` — per-caption scores, corrected text, user comment, raw errors
- `grammar_t5_model/` — T5 model + tokenizer exported via `save_pretrained()`
- `grammar_artefacts.pkl` — fixed tier thresholds, custom word list, config

## 1. Installs

In [1]:
!pip install transformers torch scikit-learn tqdm --quiet
!pip install language_tool_python --quiet
!pip install nltk --quiet
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 3.8 MB/s eta 0:00:00


True

## 2. Imports & device

In [2]:
import gc
import re
import math
import json
import difflib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForSeq2SeqLM,
    get_cosine_schedule_with_warmup,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_fscore_support, accuracy_score,
    confusion_matrix, classification_report,
)
from nltk.tokenize import sent_tokenize
from tqdm import tqdm
import language_tool_python
import warnings
warnings.filterwarnings('ignore')

device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_DIR = '/kaggle/working'
print('Using device:', device)


Using device: cuda


## 3. Configuration

In [3]:
GRAMMAR_CONFIG = {
    'correction_model': 'vennify/t5-base-grammar-correction',
    'max_input_tokens': 100,
    'max_new_tokens':   160,
    'alpha':            0.6,   # T5 similarity weight; (1-alpha) = spelling weight
}

# Word lists: local language + org terms fed directly into LanguageTool's
# spelling dictionary so they are never flagged as errors.
WORD_LIST_DIR = '/kaggle/input/datasets/mariellemayol/word-list'

# Fixed 3-tier grammar thresholds
GRAMMAR_PASS_THRESHOLD    = 0.75   # score >= 0.75  → pass
GRAMMAR_REVISION_THRESHOLD = 0.50  # 0.50 <= score < 0.75 → needs revision
                                   # score < 0.50          → fail

SEED = 42
import numpy as np, torch
torch.manual_seed(SEED)
np.random.seed(SEED)

## 4. Load & clean dataset

In [4]:
import re
import emoji
import unicodedata

def remove_unicode_font_chars(text):
    # Mathematical Bold capitals A-Z: U+1D400–U+1D419
    # Mathematical Bold smalls a-z:   U+1D41A–U+1D433
    # Mathematical Bold Italic, Script, Fraktur, etc. follow in sequence
    # Fastest approach: re-encode to ASCII, replacing anything unresolvable
    result = []
    for char in text:
        # If NFKC decomposition gives us a plain ASCII letter, use it
        normalized = unicodedata.normalize("NFKC", char)
        if normalized.isascii():
            result.append(normalized)
        else:
            # Try to get the unicode name and extract the base letter
            # e.g. "MATHEMATICAL BOLD CAPITAL A" -> "A"
            try:
                name = unicodedata.name(char, "")
                if "MATHEMATICAL" in name or "BOLD" in name or "ITALIC" in name:
                    # Last word in the name is usually the base letter/digit
                    base = name.split()[-1]
                    if len(base) == 1:
                        result.append(base)
                    # else skip it
                else:
                    result.append(char)
            except Exception:
                result.append(char)
    return "".join(result)


def clean_text(text):
    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u2019", "'").replace("\u2018", "'")
    text = text.replace("\u201c", '"').replace("\u201d", '"')
    text = text.replace("\u2013", "-").replace("\u2014", "-")
    text = text.replace("\u2026", "...")
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = emoji.replace_emoji(text, replace='')
    text = text.replace('\ufffd', '')
    text = re.sub(r"[^a-zA-Z0-9\s.,!?'\-:;()/]", ' ', text)
    # Remove apostrophes not flanked by letters on both sides
    # keeps: it's, won't, Philippines'  |  removes: ' ' ' ' isolated artifacts
    text = re.sub(r"(?<![a-zA-Z])'|'(?![a-zA-Z])", ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def clean_for_models(text):
    text = str(text)

    # 1. Remove full hashtag tokens FIRST, while # is still present
    text = re.sub(r'#\w+', '', text)

    # 2. Remove lead keywords
    text = re.sub(r'^(LOOK|READ|WATCH|CHECK THIS OUT|LISTEN|ICYMI|PSA)\s*[|:\-]?\s*',
                  '', text, flags=re.IGNORECASE)
    text = clean_text(text)
    return text


def clean_for_rules(text):
    """
    Minimal cleaning for formatting/structure rule-based features.
    Preserves |, #, capitalization patterns, lead keywords.
    """
    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u2019", "'").replace("\u2018", "'")
    text = text.replace("\u2013", "-").replace("\u2014", "-")
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = emoji.replace_emoji(text, replace='')
    text = text.replace('\ufffd', '')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [5]:
df = pd.read_csv('/kaggle/input/datasets/mariellemayol/accepted-rejected/accepted_rejected_v2.csv')
df['text']   = df['text'].astype(str).str.strip()
df['status'] = df['status'].str.lower().str.strip()
df['issue']  = df['issue'].fillna('NA').str.lower().str.strip()
df['text'] = df['text'].apply(clean_for_models)

# Deduplicate on normalised text BEFORE any split
_norm = df['text'].str.lower().str.replace(r'\s+', ' ', regex=True).str.strip()
n_before = len(df)
df = df[~_norm.duplicated(keep='first')].reset_index(drop=True)
print(f'Deduplicated: {n_before} → {len(df)} rows ({n_before - len(df)} duplicates removed)')

# Remove entries shorter than 10 characters
before_len = len(df)
df = df[df['text'].str.len() >= 10].reset_index(drop=True)
removed_short = before_len - len(df)
print(f'Short text filter: removed {removed_short} entries with < 10 characters ({before_len} → {len(df)})')

print('Dataset shape:', df.shape)
print('\nStatus distribution:')
print(df['status'].value_counts())
print('\nIssue distribution (rejected only):')
print(df[df['status'] == 'rejected']['issue'].value_counts())
display(df.head())


Deduplicated: 1822 → 1804 rows (18 duplicates removed)
Short text filter: removed 2 entries with < 10 characters (1804 → 1802)
Dataset shape: (1802, 3)

Status distribution:
status
accepted    1396
rejected     406
Name: count, dtype: int64

Issue distribution (rejected only):
issue
grammar        197
tone           105
inclusivity    104
Name: count, dtype: int64


,text,status,issue
0,ADVANCING TRANSPARENT YOUTH GOVERNANCE IN THE ...,accepted,na
1,THE CITY OF SAN JOSE DEL MONTE STRENGTHENS YOU...,accepted,na
2,Y-GOV 2026 TRAINING TO STRENGTHEN THE CAPACITY...,accepted,na
3,WE ARE HIRING! The National Youth Commission i...,accepted,na
4,SK Advisory No. 2026-NP003 : The National Yout...,accepted,na


## 5. Grammar module — T5 + LanguageTool

In [6]:
# Load grammar models

print('Loading T5 grammar correction model...')
_grammar_tokenizer = AutoTokenizer.from_pretrained(
    GRAMMAR_CONFIG['correction_model']
)
_grammar_model = AutoModelForSeq2SeqLM.from_pretrained(
    GRAMMAR_CONFIG['correction_model']
).to(device)
_grammar_model.eval()

print('Loading custom word lists...')
def _load_custom_words(word_list_dir):
    """
    Read all .txt files in word_list_dir and return a set of lowercase words.
    These are fed directly into LanguageTool's spelling dictionary so local
    language words (Tagalog, Cebuano, Ilocano, Hiligaynon, Filipino) and
    org-specific terms are never flagged as spelling errors.
    """
    from pathlib import Path
    words = set()
    wdir  = Path(word_list_dir)
    for txt_file in sorted(wdir.glob('*.txt')):
        count_before = len(words)
        for line in txt_file.read_text(encoding='utf-8', errors='ignore').splitlines():
            word = line.strip().lower()
            if word:
                words.add(word)
        print(f'  {txt_file.name}: {len(words) - count_before} words loaded')
    print(f'Total custom words: {len(words)}')
    return words

CUSTOM_WORDS = _load_custom_words(WORD_LIST_DIR)

print('Loading LanguageTool (en-US) with custom dictionary...')
_lang_tool = language_tool_python.LanguageTool(
    'en-US',
    new_spellings=list(CUSTOM_WORDS),
)
print('Grammar models ready.')


def _is_filtered_error(match, text):
    """
    Returns True if a LanguageTool match should be excluded from scoring
    and comments.

    Filters:
    - Non-spelling / non-grammar categories (WHITESPACE, STYLE, PUNCTUATION, etc.)
    - All-caps tokens (2+ chars)  → likely acronym / initialism

    Local language words (Tagalog, Cebuano, Ilocano, Hiligaynon, Filipino)
    and org-specific terms are passed to LanguageTool via new_spellings at
    init time — no corpus whitelist needed here.
    """
    # Only care about spelling (TYPOS) and grammar errors
    if match.category not in ('TYPOS', 'GRAMMAR'):
        return True

    token = text[match.offset: match.offset + match.error_length].strip()
    if not token:
        return True

    # All-caps (2+ chars) → acronym / initialism, not a typo
    if token.isupper() and len(token) > 1:
        return True

    return False


def _make_user_comment(filtered_errors, text, t5_similarity=1.0):
    """
    Convert surviving LanguageTool errors into plain-English bullet points
    for the user.  Also surfaces a T5-based notice when the correction model
    detected significant changes but LanguageTool found no specific errors
    (catches grammar restructuring that LT misses).

    Format examples:
      • Spelling: 'buyed' — did you mean 'bought'?
      • Grammar: 'She go to' — suggestion: 'She goes to'
      • No issues detected.
    """
    lines = []

    for err in filtered_errors:
        token   = text[err['offset']: err['offset'] + err['length']].strip()
        sugg    = err['suggestions']

        if err['category'] == 'TYPOS':
            if sugg:
                lines.append(
                    f"Spelling: '{token}' may be misspelled — "                    f"did you mean '{sugg[0]}'?"
                )
            else:
                lines.append(f"Spelling: '{token}' may be misspelled.")
        else:
            msg = err['message'].rstrip('.')
            if sugg:
                lines.append(f"Grammar: {msg} — suggestion: '{sugg[0]}'.")
            else:
                lines.append(f"Grammar: {msg}.")

    # Deduplicate while preserving order
    seen, deduped = set(), []
    for line in lines:
        if line not in seen:
            deduped.append(line)
            seen.add(line)

    # If T5 made substantial corrections but LT caught nothing specific,
    # add a generic notice so the comment is never misleadingly empty.
    T5_NOTICE_THRESHOLD = 0.85   # similarity below this = T5 changed a lot
    if t5_similarity < T5_NOTICE_THRESHOLD and not deduped:
        deduped.append(
            'Automated grammar check suggests this text may need review '
            '(possible grammar or phrasing issues detected).'
        )

    if not deduped:
        return 'No grammar or spelling issues detected.'

    return ' | '.join(deduped)


# Chunk splitting & T5 correction

def _split_caption_into_chunks(text, tokenizer, max_tokens):
    """
    Two-level split: paragraphs first, sentence fallback for oversized paragraphs.
    Returns list of (chunk_text, para_break_after, is_sentence).
    """
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
    chunks = []
    for para_idx, para in enumerate(paragraphs):
        token_count  = len(tokenizer.encode(para, add_special_tokens=False))
        is_last_para = (para_idx == len(paragraphs) - 1)
        if token_count <= max_tokens:
            chunks.append((para, not is_last_para, False))
        else:
            sentences = sent_tokenize(para)
            for s_idx, sent in enumerate(sentences):
                is_last_sent = (s_idx == len(sentences) - 1)
                chunks.append((sent, (is_last_sent and not is_last_para), True))
    return chunks


def _correct_chunk(chunk_text, tokenizer, model, max_input_tokens, max_new_tokens):
    """Run T5 correction on a single chunk. Returns corrected string."""
    prefix = 'gec: '
    inputs = tokenizer(
        prefix + chunk_text,
        return_tensors='pt',
        max_length=max_input_tokens + 10,
        truncation=True,
    ).to(device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            early_stopping=True,
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


def score_grammar(text):
    """
    Full grammar scoring pipeline for one caption.

    Returns:
        raw_score      float  — blended T5 similarity + spelling score [0, 1]
        corrected_text str    — full corrected caption
        filtered_errors list  — surviving LanguageTool error dicts (post-filter)
        user_comment   str    — plain-English summary for the user
    """
    cfg     = GRAMMAR_CONFIG
    alpha   = cfg['alpha']
    max_in  = cfg['max_input_tokens']
    max_new = cfg['max_new_tokens']

    # T5 correction
    chunks          = _split_caption_into_chunks(text, _grammar_tokenizer, max_in)
    corrected_parts = []
    similarities    = []

    for chunk_text, para_break_after, _ in chunks:
        corrected = _correct_chunk(chunk_text, _grammar_tokenizer,
                                   _grammar_model, max_in, max_new)
        sim = difflib.SequenceMatcher(None, chunk_text, corrected).ratio()
        similarities.append(sim)
        corrected_parts.append((corrected, para_break_after))

    # Rejoin
    rejoined, buf = [], []
    for corrected, para_break in corrected_parts:
        buf.append(corrected)
        if para_break:
            rejoined.append(' '.join(buf))
            buf = []
    if buf:
        rejoined.append(' '.join(buf))
    corrected_text = '\n\n'.join(rejoined)
    t5_similarity  = float(np.mean(similarities)) if similarities else 1.0

    # LanguageTool errors
    lt_matches = _lang_tool.check(text)
    all_errors = [
        {
            'rule_id':     m.rule_id,
            'message':     m.message,
            'offset':      m.offset,
            'length':      m.error_length,
            'context':     m.context,
            'category':    m.category,
            'suggestions': list(m.replacements[:3]),
        }
        for m in lt_matches
    ]

    # Filter errors for scoring and comments
    filtered_errors = [
        e for e, m in zip(all_errors, lt_matches)
        if not _is_filtered_error(m, text)
    ]

    # Spelling score
    spelling_errors = [e for e in filtered_errors if e['category'] == 'TYPOS']
    word_count      = max(len(text.split()), 1)
    # Normalise: each spelling error reduces score proportionally to caption length
    # ×10 scale factor so even 1 error in a short caption has visible impact
    spelling_score  = max(0.0, 1.0 - (len(spelling_errors) / word_count * 10))

    # Blended score
    raw_score    = alpha * t5_similarity + (1 - alpha) * spelling_score
    user_comment = _make_user_comment(filtered_errors, text, t5_similarity)

    return raw_score, corrected_text, filtered_errors, user_comment


# Quick smoke test
_test = "She go to the store yesterday and buyed many things. The NPA and mga members was there."
_raw, _corr, _errs, _comment = score_grammar(_test)
print(f'Input     : {_test}')
print(f'Corrected : {_corr}')
print(f'Score     : {_raw:.4f}')
print(f'Comment   : {_comment}')
print(f'Errors    : {len(_errs)}')


Loading T5 grammar correction model...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading custom word lists...
  cebuano_word_list.txt: 33693 words loaded
  custom_list.txt: 33 words loaded
  filipino_word_list.txt: 64963 words loaded
  hiligaynon_word_list.txt: 0 words loaded
  ilocano_word_list.txt: 0 words loaded
  tagalog_word_list.txt: 0 words loaded
Total custom words: 98689
Loading LanguageTool (en-US) with custom dictionary...


Grammar models ready.
Input     : She go to the store yesterday and buyed many things. The NPA and mga members was there.
Corrected : She went to the store yesterday and bought many things. The NPA and mga members were there.
Score     : 0.7040
Comment   : Grammar: The pronoun ‘She’ is usually used with a third-person or a past tense verb — suggestion: 'goes'. | Spelling: 'buyed' may be misspelled — did you mean 'bought'?
Errors    : 2


## 7. Score all captions

In [7]:
from sklearn.metrics import precision_recall_fscore_support

# Build labeled CV pool for performance evaluation
# (no threshold-selection split needed — thresholds are fixed)
def _build_grammar_df(df):
    accepted       = df[df['status'] == 'accepted'].copy()
    rejected_gram  = df[(df['status'] == 'rejected') & (df['issue'] == 'grammar')].copy()
    rejected_other = df[(df['status'] == 'rejected') & (df['issue'] != 'grammar')].copy()
    accepted['label']       = 1
    rejected_other['label'] = 1
    rejected_gram['label']  = 0
    full = pd.concat([accepted, rejected_other, rejected_gram], ignore_index=True)
    return full[['text','label']].reset_index(drop=True)

cv_df = _build_grammar_df(df)
print(f'Labeled pool for evaluation: {len(cv_df)} captions')

def _batch_score(texts):
    raws, corrs, errs, comments = [], [], [], []
    for txt in tqdm(texts, desc='grammar scoring'):
        r, c, e, cm = score_grammar(txt)
        raws.append(r); corrs.append(c); errs.append(e); comments.append(cm)
    return np.array(raws), corrs, errs, comments

print('\nScoring labeled pool (for evaluation metrics)...')
cv_raw, cv_corr, cv_errs, cv_comments = _batch_score(cv_df['text'].tolist())

print('\nScoring full dataset...')
all_raw, all_corr, all_errs, all_comments = _batch_score(df['text'].tolist())
print(f'Mean blended score: {all_raw.mean():.4f}  |  '
      f'Min: {all_raw.min():.4f}  |  Max: {all_raw.max():.4f}')


Labeled pool for evaluation: 1802 captions

Scoring labeled pool (for evaluation metrics)...


grammar scoring: 100%|██████████| 1802/1802 [2:58:23<00:00,  5.94s/it]



Scoring full dataset...


grammar scoring: 100%|██████████| 1802/1802 [2:57:53<00:00,  5.92s/it]

Mean blended score: 0.8622  |  Min: 0.4045  |  Max: 1.0000


## 8. Threshold selection

In [8]:
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
    accuracy_score,
)

def grammar_tier(score):
    """Map a raw blended grammar score (0-1) to a 3-tier decision."""
    if score >= GRAMMAR_PASS_THRESHOLD:
        return 'pass'
    elif score >= GRAMMAR_REVISION_THRESHOLD:
        return 'needs revision'
    return 'fail'

# ── Performance evaluation on the labeled pool ───────────────────────────────
# For binary metrics we collapse needs_revision + fail → 0 (not pass) vs pass → 1,
# consistent with the original label structure (accepted=1 / rejected=0).
y_true      = cv_df['label'].values
preds_tier  = np.array([grammar_tier(s) for s in cv_raw])
preds_binary = (cv_raw >= GRAMMAR_PASS_THRESHOLD).astype(int)

auc_roc = roc_auc_score(y_true, cv_raw)
auc_pr  = average_precision_score(y_true, cv_raw)
acc     = accuracy_score(y_true, preds_binary)
prec, rec, f1, _ = precision_recall_fscore_support(
    y_true, preds_binary, average='binary', zero_division=0,
)
cm = confusion_matrix(y_true, preds_binary)

print(f"\n{'='*55}")
print(f"  GRAMMAR Evaluation  (fixed 3-tier thresholds)")
print(f"{'='*55}")
print(f'  Pass threshold     : >= {GRAMMAR_PASS_THRESHOLD}')
print(f'  Needs revision     : >= {GRAMMAR_REVISION_THRESHOLD} and < {GRAMMAR_PASS_THRESHOLD}')
print(f'  Fail threshold     : <  {GRAMMAR_REVISION_THRESHOLD}')
print(f"  {'─'*45}")
print(f'  AUC-ROC : {auc_roc:.4f}  |  AUC-PR   : {auc_pr:.4f}')
print(f'  Accuracy: {acc:.4f}  |  Precision: {prec:.4f}')
print(f'  Recall  : {rec:.4f}  |  F1        : {f1:.4f}')
print(f'  Confusion matrix (pass vs not-pass):')
print(f'    [[TN={cm[0,0]} FP={cm[0,1]}]')
print(f'     [FN={cm[1,0]} TP={cm[1,1]}]]')
print()
print(classification_report(
    y_true, preds_binary,
    target_names=['not pass', 'pass'], zero_division=0,
))

# Tier distribution on the labeled pool
n = len(preds_tier)
for tier in ['pass', 'needs revision', 'fail']:
    cnt = (preds_tier == tier).sum()
    print(f'  {tier:<20s}: {cnt:>4d}  ({cnt/n*100:.1f}%)')



  GRAMMAR Evaluation  (fixed 3-tier thresholds)
  Pass threshold     : >= 0.75
  Needs revision     : >= 0.5 and < 0.75
  Fail threshold     : <  0.5
  ─────────────────────────────────────────────
  AUC-ROC : 0.7450  |  AUC-PR   : 0.9613
  Accuracy: 0.8235  |  Precision: 0.9133
  Recall  : 0.8860  |  F1        : 0.8994
  Confusion matrix (pass vs not-pass):
    [[TN=62 FP=135]
     [FN=183 TP=1422]]

              precision    recall  f1-score   support

    not pass       0.25      0.31      0.28       197
        pass       0.91      0.89      0.90      1605

    accuracy                           0.82      1802
   macro avg       0.58      0.60      0.59      1802
weighted avg       0.84      0.82      0.83      1802

  pass                : 1557  (86.4%)
  needs revision      :  241  (13.4%)
  fail                :    4  (0.2%)


## 9. Save scores & model

In [9]:
import pickle, os

df['grammar_raw_score'] = all_raw.round(4)
df['grammar_score']     = df['grammar_raw_score'] 
df['grammar_decision']  = [grammar_tier(p) for p in all_raw]
df['grammar_corrected'] = all_corr
df['grammar_comment']   = all_comments
df['grammar_lt_errors'] = [json.dumps(e) for e in all_errs]

out_cols = ['text','status','issue',
            'grammar_raw_score','grammar_score','grammar_decision',
            'grammar_corrected','grammar_comment','grammar_lt_errors']
df[out_cols].to_csv(f'{SAVE_DIR}/grammar_scores.csv', index=False)
print(f'Saved → {SAVE_DIR}/grammar_scores.csv')
display(df[out_cols][['text','grammar_score','grammar_decision','grammar_comment']].head(5))

# Save T5 model for deployment
grammar_model_dir = f'{SAVE_DIR}/grammar_t5_model'
os.makedirs(grammar_model_dir, exist_ok=True)
_grammar_model.save_pretrained(grammar_model_dir)
_grammar_tokenizer.save_pretrained(grammar_model_dir)
print(f'Saved T5 model → {grammar_model_dir}')

# Save artefacts for Notebook 3
artefacts = {
    'pass_threshold':     GRAMMAR_PASS_THRESHOLD,
    'revision_threshold': GRAMMAR_REVISION_THRESHOLD,
    'custom_words':       CUSTOM_WORDS,
    'config':             GRAMMAR_CONFIG,
    'grammar_model_dir':  grammar_model_dir,
    'grammar_tier':       grammar_tier,
}
with open(f'{SAVE_DIR}/grammar_artefacts.pkl', 'wb') as f:
    pickle.dump(artefacts, f)
print(f'Saved artefacts → {SAVE_DIR}/grammar_artefacts.pkl')

print('\nFiles in SAVE_DIR:')
for fn in sorted(os.listdir(SAVE_DIR)):
    print(f'  {fn:50s}  {os.path.getsize(f"{SAVE_DIR}/{fn}")/1e6:6.1f} MB')


Saved → /kaggle/working/grammar_scores.csv


,text,grammar_score,grammar_decision,grammar_comment
0,ADVANCING TRANSPARENT YOUTH GOVERNANCE IN THE ...,0.8906,pass,Spelling: 'Joie' may be misspelled — did you m...
1,THE CITY OF SAN JOSE DEL MONTE STRENGTHENS YOU...,0.8825,pass,Spelling: 'Bulacan' may be misspelled — did yo...
2,Y-GOV 2026 TRAINING TO STRENGTHEN THE CAPACITY...,0.7646,pass,Spelling: 'Candaba' may be misspelled — did yo...
3,WE ARE HIRING! The National Youth Commission i...,0.8874,pass,Grammar: A verb or adverb may be missing or mi...
4,SK Advisory No. 2026-NP003 : The National Yout...,0.7907,pass,Spelling: 'LYDOs' may be misspelled — did you ...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved T5 model → /kaggle/working/grammar_t5_model
Saved artefacts → /kaggle/working/grammar_artefacts.pkl

Files in SAVE_DIR:
  __notebook__.ipynb                                     0.9 MB
  grammar_artefacts.pkl                                  1.2 MB
  grammar_scores.csv                                     8.4 MB
  grammar_t5_model                                       0.0 MB
